In [5]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# For formatting
BOLD = '\033[1m'
END = '\033[0m'

In [12]:
# Read in dat file into dataframe
# Column names
column_names = [
    'Battle',
    'Year',
    'Portuguese_ships',
    'Dutch_ships',
    'English_ships',
    'Ratio_Portuguese_to_Dutch_British',
    'Spanish_involvement',
    'Portuguese_outcome'
]

df = pd.read_fwf('armada.dat', names = column_names)
df

,Battle,Year,Portuguese_ships,Dutch_ships,English_ships,Ratio_Portuguese_to_Dutch_British,Spanish_involvement,Portuguese_outcome
0,Bantam,1601,6,3,0,2.000,0,0
1,Malacca Strait,1606,14,11,0,1.273,0,0
2,Ilha das Naus,1606,6,9,0,0.667,0,-1
3,Pulo Butum,1606,7,9,0,0.778,0,1
4,Surrat,1615,6,0,4,1.500,0,0
5,Ilha das Naus,1615,3,5,0,0.600,0,-1
6,Jask,1620,4,0,4,1.000,0,0
7,Hormuz,1622,6,0,5,1.200,0,-1
8,Mogincoal Shoals,1622,4,4,2,0.667,0,-1
9,Hormuz,1625,8,4,4,1.000,0,0


In [7]:
df.head()

,Battle,Year,Portuguese_ships,Dutch_ships,English_ships,Ratio_Portuguese_to_Dutch_British,Spanish_involvement,Portuguese_outcome
0,Bantam,1601,6,3,0,2.000,0,0
1,Malacca Strait,1606,14,11,0,1.273,0,0
2,Ilha das Naus,1606,6,9,0,0.667,0,-1
3,Pulo Butum,1606,7,9,0,0.778,0,1
4,Surrat,1615,6,0,4,1.500,0,0


Instructions

    Use an SVM-based model to predict the Portuguese outcome of the battle from the number of ships involved on all sides and Spanish involvement.

    Try solving the same problem using two other classifiers that you know.

    Report and compare their results with those from SVM.


In [8]:
# SVM based model approach to predict Portuguese outcome (y with -1, 0 or 1) using number of ships involved on all sides and Spanish involvement

# Prepare features and y
X = df[['Portuguese_ships', 'Dutch_ships', 'English_ships', 'Spanish_involvement']]
y = df['Portuguese_outcome']
iterations = 1000

# Create function to run iterations for an average accuracy test
def accuracy_test(model_type, X, y, iterations):
    # Init parameters
    test_accuracy = []
    train_accuracy = []
    model = None
    
    for i in range(iterations):
        # Split data (70% train, 30% test)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3)

        if model_type == SVC:
            # Use rbf as a safe default
            model = model_type(kernel = 'rbf') # https://www.geeksforgeeks.org/machine-learning/how-to-choose-the-best-kernel-function-for-svms/
        elif model_type == LogisticRegression:
            model = model_type(max_iter = 1000)
        else:
            model = model_type()
        
        # Fit the model
        model.fit(X_train, y_train)
        y_pred_test = model.predict(X_test)
        y_pred_train = model.predict(X_train) 
        test_accuracy.append(accuracy_score(y_test, y_pred_test))
        train_accuracy.append(accuracy_score(y_train,y_pred_train))

        # Calculate average accuracies
        avg_acc_test = sum(test_accuracy)/len(test_accuracy)
        avg_acc_train = sum(train_accuracy)/len(train_accuracy)
    return avg_acc_test, avg_acc_train

In [9]:
# Run SVM model and get accuracies
svc_test_acc, svc_train_acc = accuracy_test(SVC, X, y, iterations)

print(f'{BOLD}Support Vector Machine Model Accuracies{END}')
print(f'Average test accuracy over {iterations} iterations: {svc_test_acc * 100:.2f}%')
print(f'Average training accuracy over {iterations} iterations: {svc_train_acc * 100:.2f}%')

Support Vector Machine Model Accuracies
Average test accuracy over 1000 iterations: 40.77%
Average training accuracy over 1000 iterations: 58.04%


In [10]:
# Run tree model and get accuracies
tree_test_acc, tree_train_acc = accuracy_test(RandomForestClassifier, X, y, iterations)

print(f'{BOLD}Random Forest Accuracies{END}')
print(f'Average test accuracy over {iterations} iterations: {tree_test_acc * 100:.2f}%')
print(f'Average training accuracy over {iterations} iterations: {tree_train_acc * 100:.2f}%')

Random Forest Accuracies
Average test accuracy over 1000 iterations: 31.17%
Average training accuracy over 1000 iterations: 100.00%


In [11]:
# Run logistic regression model and get accuracies
log_test_acc, log_train_acc = accuracy_test(LogisticRegression, X, y, iterations)

print(f'{BOLD}Logistic Regression Accuracies{END}')
print(f'Average test accuracy over {iterations} iterations: {log_test_acc * 100:.2f}%')
print(f'Average training accuracy over {iterations} iterations: {log_train_acc * 100:.2f}%')

Logistic Regression Accuracies
Average test accuracy over 1000 iterations: 37.96%
Average training accuracy over 1000 iterations: 65.35%


### Analysis
SVM consistently has a higher test accuracy in the 40% - 45% range, performing better than the random forest and logistic regression classifiers with the test dataset. Interestingly, the random forest classifier performs very well in the training accuracies, much better than the other two models, which range around 50% - 60% accuracies.